In [1]:
!pip install torch transformers scikit-learn pandas tqdm nltk

In [13]:
import os, re, random, csv, torch, torch.nn as nn, pandas as pd
from datetime import datetime, timedelta
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from tqdm import tqdm


In [41]:
PROJECT_ROOT = "./NLP_Project"

os.makedirs(f"NLP FA2/data", exist_ok=True)
os.makedirs(f"NLP FA2/saved_model", exist_ok=True)

DATA_PATH = f"NLP FA2/data/grievances.csv"
SAVE_PATH = f"NLP FA2/saved_model/model.pt"

DEVICE = torch.device("cpu")

BATCH_SIZE = 8
NUM_EPOCHS = 2   
LR = 2e-5
MAX_LEN = 64

print("Device:", DEVICE)

Device: cpu


In [43]:
ID2LABEL = {
    0: "Consumer Law",
    1: "Cyber Crime",
    2: "Property Disputes",
    3: "Domestic Violence",
    4: "Labour Law",
    5: "Environmental Law",
}
LABEL2ID = {v:k for k,v in ID2LABEL.items()}
NUM_CLASSES = len(ID2LABEL)

In [45]:
def add_context_noise(text):
    extras = [
        "this is very serious",
        "please help me",
        "not sure what to do",
        "this has been going on for weeks",
        "i am really stressed",
        "need urgent help",
        "very frustrating situation"
    ]
    
    if random.random() < 0.5:
        text += " " + random.choice(extras)
    
    return text

TRAIN_TEMPLATES = {
    "Cyber Crime": [
        "someone accessed my bank account without permission",
        "unauthorized transaction happened from my account",
        "i lost money due to an online scam",
    ],
    "Consumer Law": [
        "i bought a product but it was defective",
        "the company refused to refund my money",
        "i received a damaged item from an online store",
    ],
    "Labour Law": [
        "my employer has not paid my salary",
        "i am being forced to work extra hours without pay",
        "i was removed from my job unfairly",
    ],
    "Domestic Violence": [
        "my husband physically assaulted me",
        "i am facing abuse at home",
        "i feel unsafe living with my family",
    ],
    "Property Disputes": [
        "there is a dispute over my land ownership",
        "the builder is not giving possession of my flat",
    ],
    "Environmental Law": [
        "a factory near my home is causing pollution",
        "garbage is being dumped near my house",
    ],
}


TEST_TEMPLATES = {
    "Cyber Crime": [
    "unauthorized activity happened in my account",
    "money was taken from my account unexpectedly",
    "i noticed suspicious transactions recently",
    ],
    "Consumer Law": [
        "i was sold a faulty electronic item",
        "the seller is not honoring the warranty",
    ],
    "Labour Law": [
    "i am not receiving my payments properly",
    "my company is not treating me fairly",
    "i am facing issues at my workplace",
    ],
    "Domestic Violence": [
        "i am being mistreated by my spouse",
        "there is violence happening in my home",
    ],
    "Property Disputes": [
        "someone illegally occupied my property",
        "ownership conflict over my house",
    ],
    "Environmental Law": [
        "industrial waste is polluting nearby water",
        "air pollution from factory is affecting health",
    ],
}


import csv

TOTAL_SAMPLES = 2000
NUM_CATEGORIES = len(TRAIN_TEMPLATES)

samples_per_category = TOTAL_SAMPLES // NUM_CATEGORIES
AMBIGUOUS_SENTENCES = [
    "i lost money but i dont know how",
    "someone took my money",
    "i am facing issues with payment",
    "i am having problems with a service",
    "something went wrong with my transaction",
    "i am being treated unfairly",
]

def generate_dataset(templates, n_per_category):
    data = []
    
    for category in templates:
        for _ in range(n_per_category):
            
            # 🔥 70% normal, 30% ambiguous
            if random.random() < 0.85:
                text = random.choice(templates[category])
            else:
                text = random.choice(AMBIGUOUS_SENTENCES)
            
            text = add_context_noise(text)
            if random.random() < 0.15:
                    text += " " + random.choice([
                        "i went to office yesterday",
                        "this happened suddenly",
                        "i spoke to someone about this",
                        "not sure who is responsible"
                    ])

            # 🔥 10% label noise
            if random.random() < 0.05:
                wrong_category = random.choice(list(templates.keys()))
                label = wrong_category
            else:
                label = category
            
            data.append({
                "complaint": text,
                "category": label
            })
    
    return data

# Generate dataset
data = generate_dataset(TRAIN_TEMPLATES, samples_per_category)

# If total not exactly 2000, add remaining
while len(data) < TOTAL_SAMPLES:
    category = random.choice(list(TRAIN_TEMPLATES.keys()))
    text = random.choice(TRAIN_TEMPLATES[category])
    text = add_context_noise(text)
    
    data.append({
        "complaint": text,
        "category": category
    })


# 🔥 SAVE TO CSV
with open(DATA_PATH, "w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=["complaint", "category"])
    writer.writeheader()
    writer.writerows(data)

print(f"✅ Dataset saved at: {DATA_PATH}")
print(f"Total samples: {len(data)}")

import pandas as pd
from sklearn.model_selection import train_test_split

df = pd.read_csv(DATA_PATH)

texts = df["complaint"].tolist()
labels = df["category"].map(LABEL2ID).tolist()

train_texts, test_texts, train_labels, test_labels = train_test_split(
    texts, labels, test_size=0.2, random_state=42, stratify=labels
)

print("Train size:", len(train_texts))
print("Test size:", len(test_texts))

✅ Dataset saved at: NLP FA2/data/grievances.csv
Total samples: 2000
Train size: 1600
Test size: 400


In [47]:
from transformers import DistilBertTokenizerFast
tokenizer = DistilBertTokenizerFast.from_pretrained("distilbert-base-uncased")

In [48]:
class GrievanceDataset(Dataset):
    def __init__(self, texts, labels):
        self.texts = texts
        self.labels = labels

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        enc = tokenizer(
            self.texts[idx],
            padding="max_length",
            truncation=True,
            max_length=MAX_LEN,
            return_tensors="pt"
        )

        return {
            "input_ids": enc["input_ids"].squeeze(),
            "attention_mask": enc["attention_mask"].squeeze(),
           "label": torch.tensor(self.labels[idx], dtype=torch.long)
        }

In [51]:
train_ds = GrievanceDataset(train_texts, train_labels)
test_ds  = GrievanceDataset(test_texts, test_labels)

train_dl = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
test_dl  = DataLoader(test_ds, batch_size=BATCH_SIZE)

In [53]:
from transformers import DistilBertModel

class Model(nn.Module):
    def __init__(self):
        super().__init__()
        self.bert = DistilBertModel.from_pretrained("distilbert-base-uncased")
        self.dropout = nn.Dropout(0.5)
        self.fc = nn.Linear(768, NUM_CLASSES)

    def forward(self, input_ids, attention_mask):
        out = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        x = out.last_hidden_state[:, 0]
        x = self.dropout(x)
        return self.fc(x)

In [55]:
model = Model().to(DEVICE)
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=LR)

def run_epoch(loader, train=True):
    model.train() if train else model.eval()

    total_loss, correct, total = 0, 0, 0

    with torch.set_grad_enabled(train):
        for batch in tqdm(loader):
            ids = batch["input_ids"].to(DEVICE)
            mask = batch["attention_mask"].to(DEVICE)
            labels = batch["label"].to(DEVICE)

            if train:
                optimizer.zero_grad()

            outputs = model(ids, mask)
            loss = loss_fn(outputs, labels)

            if train:
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step()

            total_loss += loss.item()
            preds = torch.argmax(outputs, dim=1)
            correct += (preds == labels).sum().item()
            total += len(labels)

    return total_loss / len(loader), correct / total


for epoch in range(NUM_EPOCHS):
    print(f"\nEpoch {epoch+1}")
    tr_loss, tr_acc = run_epoch(train_dl, True)
    te_loss, te_acc = run_epoch(test_dl, False)

    print("Train Acc:", tr_acc)
    print("Test Acc:", te_acc)


Epoch 1


100%|██████████████████████████████████████████████████████████████████████████████████| 50/50 [00:14<00:00,  3.41it/s]


Train Acc: 0.72625
Test Acc: 0.825

Epoch 2


100%|██████████████████████████████████████████████████████████████████████████████████| 50/50 [00:15<00:00,  3.29it/s]

Train Acc: 0.836875
Test Acc: 0.8475


In [57]:
torch.save(model.state_dict(), SAVE_PATH)
tokenizer.save_pretrained("./tokenizer")

('./tokenizer\\tokenizer_config.json',
 './tokenizer\\special_tokens_map.json',
 './tokenizer\\vocab.txt',
 './tokenizer\\added_tokens.json',
 './tokenizer\\tokenizer.json')

In [72]:
!pip install deep-translator langdetect

In [3]:
from deep_translator import GoogleTranslator
from langdetect import detect

In [63]:
### Model loading ##
import torch
import torch.nn as nn
from transformers import DistilBertTokenizerFast, DistilBertModel

DEVICE = torch.device("cpu")

SAVE_PATH = "./saved_models/saved_model/model.pt"
TOKENIZER_PATH = "./tokenizer"

In [65]:
ID2LABEL = {
    0: "Consumer Law",
    1: "Cyber Crime",
    2: "Property Disputes",
    3: "Domestic Violence",
    4: "Labour Law",
    5: "Environmental Law",
}
NUM_CLASSES = len(ID2LABEL)

In [67]:
class Model(nn.Module):
    def __init__(self):
        super().__init__()
        self.bert = DistilBertModel.from_pretrained("distilbert-base-uncased")
        self.dropout = nn.Dropout(0.5)
        self.fc = nn.Linear(768, NUM_CLASSES)

    def forward(self, input_ids, attention_mask):
        out = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        x = out.last_hidden_state[:, 0]
        x = self.dropout(x)
        return self.fc(x)

In [69]:
# Load tokenizer
tokenizer = DistilBertTokenizerFast.from_pretrained(TOKENIZER_PATH)

# Load model
model = Model()
model.load_state_dict(torch.load(SAVE_PATH, map_location=DEVICE))
model.to(DEVICE)
model.eval()

print("✅ Model loaded successfully!")

✅ Model loaded successfully!


In [70]:
from deep_translator import GoogleTranslator

def translate_to_english(text):
    try:
        translated = GoogleTranslator(source='auto', target='en').translate(text)
        return translated, "auto"
    except:
        return text, "unknown"

In [71]:
def predict(text):
    translated_text, lang = translate_to_english(text)
    
    enc = tokenizer(
        translated_text,
        return_tensors="pt",
        truncation=True,
        padding=True,
        max_length=64
    )
    
    ids = enc["input_ids"].to(DEVICE)
    mask = enc["attention_mask"].to(DEVICE)
    
    with torch.no_grad():
        outputs = model(ids, mask)
        probs = torch.softmax(outputs, dim=1)
        pred = torch.argmax(probs).item()
    
    return {
        "original_text": text,
        "translated_text": translated_text,
        "prediction": ID2LABEL[pred],
        "confidence": float(probs[0][pred])
    }

In [75]:
print(predict("mera account hack ho gaya"))
print(predict("my company is not paying salary"))

{'original_text': 'mera account hack ho gaya', 'translated_text': 'my account got hacked', 'prediction': 'Cyber Crime', 'confidence': 0.9740831255912781}
{'original_text': 'my company is not paying salary', 'translated_text': 'my company is not paying salary', 'prediction': 'Labour Law', 'confidence': 0.8633549809455872}


In [48]:
print(predict("मेरे पति मुझे रोज मारते हैं"))

{'original_text': 'मेरे पति मुझे रोज मारते हैं', 'translated_text': 'my husband beats me every day', 'prediction': 'Domestic Violence', 'confidence': 0.9310950636863708}


In [51]:
print(predict("factory causing pollution near my house"))

{'original_text': 'factory causing pollution near my house', 'translated_text': 'factory causing pollution near my house', 'prediction': 'Environmental Law', 'confidence': 0.9546692371368408}


In [53]:
print(predict("i am facing issues at my workplace and treated unfairly"))

{'original_text': 'i am facing issues at my workplace and treated unfairly', 'translated_text': 'i am facing issues at my workplace and treated unfairly', 'prediction': 'Labour Law', 'confidence': 0.7592366933822632}


In [55]:
print(predict("i feel unsafe at home and scared"))

{'original_text': 'i feel unsafe at home and scared', 'translated_text': 'i feel unsafe at home and scared', 'prediction': 'Domestic Violence', 'confidence': 0.9753977656364441}


In [57]:
print(predict("mera paisa kisi ne online chura liya"))

{'original_text': 'mera paisa kisi ne online chura liya', 'translated_text': 'someone stole my money online', 'prediction': 'Cyber Crime', 'confidence': 0.46478700637817383}


In [61]:
print(predict("माझ्या घराजवळ कारखाना प्रदूषण करत आहे"))
print(predict("મારા બેંક ખાતામાંથી પૈસા ચોરી ગયા"))

{'original_text': 'माझ्या घराजवळ कारखाना प्रदूषण करत आहे', 'translated_text': 'A factory is polluting near my house', 'prediction': 'Environmental Law', 'confidence': 0.9671316742897034}
{'original_text': 'મારા બેંક ખાતામાંથી પૈસા ચોરી ગયા', 'translated_text': 'Money was stolen from my bank account', 'prediction': 'Cyber Crime', 'confidence': 0.9044816493988037}
